# BÀI TẬP CHƯƠNG 2



## Câu 1 : Hãy kết nối hai bảng trên theo những cách sau: 
- Sử dụng tích Decartes. 
- Sử dụng JOIN: INNER JOIN, LEFT JOIN, RIGHT JOIN, FULL OUTER JOIN.

In [1]:
import pandas as pd
import numpy as np
import sqlite3

# 1. Khởi tạo dữ liệu (giữ nguyên dữ liệu của bạn)
student_data = {
    'student_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'name': ['Nguyen Minh Hoang', 'Tran Thi Lan', 'Pham Van Nam', 'Le Thanh Huyen', 
             'Vu Quoc Anh', 'Dang Thuy Linh', 'Bui Tien Dung', 'Ho Ngoc Mai', 
             'Duong Huu Phuc', 'Cao Thi Hanh'],
    'class': ['May Tinh', 'Kinh Te', 'Toan Tin', 'Toan Tin', 'May Tinh', 
              'May Tinh', 'Kinh Te', 'Toan Tin', 'Kinh Te', 'May Tinh'],
    'course_id': [12, 34, np.nan, 20, 24, 24, 34, 20, np.nan, np.nan],
    'score': [6.7, 9.2, 7.9, 7.2, 8.0, 5.5, 9.2, 8.8, 7.2, 7.0]
}

course_data = {
    'id': [12, 34, 26],
    'course_name': ['Giai tich', 'Thong ke', 'Tin hoc']
}

df_student = pd.DataFrame(student_data)
df_course = pd.DataFrame(course_data)

# 2. Tạo kết nối SQL trong bộ nhớ (In-memory database)
conn = sqlite3.connect(':memory:')

# Đẩy dữ liệu từ DataFrame vào SQL tables
df_student.to_sql('students', conn, index=False)
df_course.to_sql('courses', conn, index=False)

# Hàm hỗ trợ chạy SQL và trả về DataFrame
def run_sql(query):
    return pd.read_sql_query(query, conn)

# 1. Tích Descartes (Cross Join)
sql_cross = "SELECT * FROM students CROSS JOIN courses"
descartes = run_sql(sql_cross)

# 2. INNER JOIN
sql_inner = """
SELECT * FROM students 
INNER JOIN courses ON students.course_id = courses.id
"""
inner_join = run_sql(sql_inner)

# 3. LEFT JOIN
sql_left = """
SELECT * FROM students 
LEFT JOIN courses ON students.course_id = courses.id
"""
left_join = run_sql(sql_left)

# 4. RIGHT JOIN 
# Lưu ý: SQLite tiêu chuẩn không hỗ trợ RIGHT JOIN trực tiếp. 
# Ta giả lập bằng cách đổi vị trí bảng trong LEFT JOIN.
sql_right = """
SELECT * FROM courses 
LEFT JOIN students ON courses.id = students.course_id
"""
right_join = run_sql(sql_right)

# 5. FULL OUTER JOIN
# Lưu ý: SQLite không hỗ trợ FULL OUTER JOIN. 
# Ta giả lập bằng cách UNION giữa LEFT JOIN và "RIGHT JOIN".
sql_full_outer = """
SELECT * FROM students LEFT JOIN courses ON students.course_id = courses.id
UNION ALL
SELECT * FROM courses LEFT JOIN students ON courses.id = students.course_id
WHERE students.course_id IS NULL
"""
full_outer_join = run_sql(sql_full_outer)

# Hiển thị kết quả mẫu
print("Kết quả Inner Join (SQL):")
print(inner_join)

Kết quả Inner Join (SQL):
   student_id               name     class  course_id  score  id course_name
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7  12   Giai tich
1           2       Tran Thi Lan   Kinh Te       34.0    9.2  34    Thong ke
2           7      Bui Tien Dung   Kinh Te       34.0    9.2  34    Thong ke


## Câu 2 : Hãy cập nhật những giá trị course_id còn thiếu trong bảng student bằng câu lệnh SQL, trong đó các giá trị được điền là những giá trị nằm trong bảng course  và loại bỏ những bản ghi tham gia những môn học không tồn tại bảng course. Sau đó hãy cho biết: 
- a. Tổng số sinh viên, điểm trung bình của từng lớp. 
- b. Tổng số sinh viên, điểm trung bình của từng môn học. 
- c. Phân loại thi đua theo số điểm của từng môn học biết: 
>> . Điểm TB ≥  9.0: Xuất sắc. 
>> ii. 6.0 ≤ Điểm TB ≤ 8.9: Tốt. 
>> iii. Điểm TB < 6.0: Kém. 
3. Hãy xếp hạng sinh viên thông qua: 
- a. Điểm số. 
- b. Điểm số theo lớp học. 
- c. Điểm số theo mã môn học. 
- và cho biết top 3 sinh viện đạt thứ hạng cao nhất, top 3 sinh viên đạt thứ hạng thấp nhất theo từng trường hợp trên. 
4. Hãy bổ sung thêm một trường graduation_date có kiểu dữ liệu là DATETIME vào bảng 
student để xác định thời gian tốt nghiệp của từng bạn, trong đó thời gian tốt nghiệp được 
xác định bởi thời gian hiện tại cộng với số hạng tương ứng của bạn đó tính theo điểm số. 
 

In [3]:
# 1. Tạo bảng df_student_clean 
# Lọc: lấy những course_id có trong bảng courses HOẶC course_id bị NULL
# Sau đó điền NULL = 26
sql_clean_data = """
CREATE TABLE student_clean AS
SELECT 
    student_id, 
    name, 
    class, 
    COALESCE(course_id, 26) AS course_id, 
    score
FROM students
WHERE course_id IN (SELECT id FROM courses) 
   OR course_id IS NULL;
"""
conn.execute("DROP TABLE IF EXISTS student_clean") # Xóa nếu đã tồn tại
conn.execute(sql_clean_data)

# 2. Tạo bảng final bằng cách JOIN với bảng courses
sql_final = """
CREATE TABLE df_final AS
SELECT s.*, c.course_name
FROM student_clean s
INNER JOIN courses c ON s.course_id = c.id
"""
conn.execute("DROP TABLE IF EXISTS df_final")
conn.execute(sql_final)

print("Dữ liệu sau khi làm sạch (df_final):")
display(pd.read_sql("SELECT * FROM df_final", conn))

#2. Thống kê theo lớp và môn học
# a. Thống kê theo Lớp
sql_stats_class = """
SELECT 
    class,
    COUNT(student_id) AS Tong_SV,
    ROUND(AVG(score), 2) AS Diem_TB
FROM df_final
GROUP BY class
"""
print("\na. Thống kê theo Lớp:")
display(pd.read_sql(sql_stats_class, conn))

# b. Thống kê theo Môn học
sql_stats_course = """
SELECT 
    course_name,
    COUNT(student_id) AS Tong_SV,
    ROUND(AVG(score), 2) AS Diem_TB
FROM df_final
GROUP BY course_name
"""
print("\nb. Thống kê theo Môn học:")
stats_by_course = pd.read_sql(sql_stats_course, conn)
display(stats_by_course)
# 3. Phân loại thi đua

sql_classification = """
SELECT 
    course_name,
    Diem_TB,
    CASE 
        WHEN Diem_TB >= 9.0 THEN 'Xuất sắc'
        WHEN Diem_TB BETWEEN 6.0 AND 8.9 THEN 'Tốt'
        ELSE 'Kém'
    END AS Xep_Loai
FROM (
    SELECT 
        course_name,
        ROUND(AVG(score), 2) AS Diem_TB
    FROM df_final
    GROUP BY course_name
)
"""
print("\nc. Phân loại thi đua theo Môn học:")
display(pd.read_sql(sql_classification, conn))

Dữ liệu sau khi làm sạch (df_final):


,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,3,Pham Van Nam,Toan Tin,26.0,7.9,Tin hoc
3,7,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
4,9,Duong Huu Phuc,Kinh Te,26.0,7.2,Tin hoc
5,10,Cao Thi Hanh,May Tinh,26.0,7.0,Tin hoc



a. Thống kê theo Lớp:


,class,Tong_SV,Diem_TB
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90



b. Thống kê theo Môn học:


,course_name,Tong_SV,Diem_TB
0,Giai tich,1,6.70
1,Thong ke,2,9.20
2,Tin hoc,3,7.37



c. Phân loại thi đua theo Môn học:


,course_name,Diem_TB,Xep_Loai
0,Giai tich,6.70,Tốt
1,Thong ke,9.20,Xuất sắc
2,Tin hoc,7.37,Tốt


## Câu 3 : Hãy xếp hạng sinh viên thông qua: 
- a. Điểm số. 
- b. Điểm số theo lớp học. 
- c. Điểm số theo mã môn học. 
- và cho biết top 3 sinh viện đạt thứ hạng cao nhất, top 3 sinh viên đạt thứ hạng thấp nhất theo từng trường hợp trên.

In [4]:
# a. Xếp hạng theo Điểm số tổng thể
sql_overall_rank = """
SELECT 
    student_id, name, class, score,
    RANK() OVER (ORDER BY score DESC) as Rank_Overall
FROM students
"""
df_overall = pd.read_sql(sql_overall_rank, conn)

print("Top 3 sinh viên điểm CAO nhất toàn trường:")
display(df_overall.nsmallest(3, 'Rank_Overall')) # Hoặc dùng SQL LIMIT 3

# b. Xếp hạng điểm số theo lớp học
sql_class_rank = """
SELECT 
    student_id, name, class, score,
    RANK() OVER (PARTITION BY class ORDER BY score DESC) as Rank_Class
FROM students
ORDER BY class, Rank_Class
"""
df_class = pd.read_sql(sql_class_rank, conn)

print("\nTop 3 sinh viên điểm CAO nhất theo TỪNG LỚP:")
# Lọc lấy những người có hạng <= 3 trong mỗi lớp
display(df_class[df_class['Rank_Class'] <= 3])

# c. Xếp hạng điểm số theo mã môn học (course_id)

sql_course_rank = """
SELECT 
    student_id, name, course_id, score,
    RANK() OVER (PARTITION BY course_id ORDER BY score DESC) as Rank_Course
FROM students
ORDER BY course_id, Rank_Course
"""
df_course_ranked = pd.read_sql(sql_course_rank, conn)

print("\nTop 3 sinh viên điểm CAO nhất theo TỪNG MÔN:")
display(df_course_ranked[df_course_ranked['Rank_Course'] <= 3])

Top 3 sinh viên điểm CAO nhất toàn trường:


,student_id,name,class,score,Rank_Overall
0,2,Tran Thi Lan,Kinh Te,9.2,1
1,7,Bui Tien Dung,Kinh Te,9.2,1
2,8,Ho Ngoc Mai,Toan Tin,8.8,3



Top 3 sinh viên điểm CAO nhất theo TỪNG LỚP:


,student_id,name,class,score,Rank_Class
0,2,Tran Thi Lan,Kinh Te,9.2,1
1,7,Bui Tien Dung,Kinh Te,9.2,1
2,9,Duong Huu Phuc,Kinh Te,7.2,3
3,5,Vu Quoc Anh,May Tinh,8.0,1
4,10,Cao Thi Hanh,May Tinh,7.0,2
5,1,Nguyen Minh Hoang,May Tinh,6.7,3
7,8,Ho Ngoc Mai,Toan Tin,8.8,1
8,3,Pham Van Nam,Toan Tin,7.9,2
9,4,Le Thanh Huyen,Toan Tin,7.2,3



Top 3 sinh viên điểm CAO nhất theo TỪNG MÔN:


,student_id,name,course_id,score,Rank_Course
0,3,Pham Van Nam,NaN,7.9,1
1,9,Duong Huu Phuc,NaN,7.2,2
2,10,Cao Thi Hanh,NaN,7.0,3
3,1,Nguyen Minh Hoang,12.0,6.7,1
4,8,Ho Ngoc Mai,20.0,8.8,1
5,4,Le Thanh Huyen,20.0,7.2,2
6,5,Vu Quoc Anh,24.0,8.0,1
7,6,Dang Thuy Linh,24.0,5.5,2
8,2,Tran Thi Lan,34.0,9.2,1
9,7,Bui Tien Dung,34.0,9.2,1


## Câu 4 :  Hãy bổ sung thêm một trường graduation_date có kiểu dữ liệu là DATETIME vào bảng student để xác định thời gian tốt nghiệp của từng bạn, trong đó thời gian tốt nghiệp được xác định bởi thời gian hiện tại cộng với số hạng tương ứng của bạn đó tính theo điểm số.

In [5]:
sql_graduation = """
SELECT 
    student_id, 
    name, 
    score,
    -- 1. Tính thứ hạng (Rank)
    RANK() OVER (ORDER BY score DESC) as Rank_Overall,
    -- 2. Lấy thời gian hiện tại
    DATETIME('now', 'localtime') as current_time_ref,
    -- 3. Cộng số ngày tương ứng với thứ hạng vào ngày hiện tại
    -- Trong SQLite, ta dùng hàm DATE('now', '+' || rank || ' days')
    DATE('now', 'localtime', '+' || (RANK() OVER (ORDER BY score DESC)) || ' days') as graduation_date
FROM students
ORDER BY Rank_Overall;
"""

df_graduation = pd.read_sql(sql_graduation, conn)

print("Bảng danh sách sinh viên cùng ngày tốt nghiệp dự kiến (SQL):")
display(df_graduation[['student_id', 'name', 'score', 'Rank_Overall', 'graduation_date']])

Bảng danh sách sinh viên cùng ngày tốt nghiệp dự kiến (SQL):


,student_id,name,score,Rank_Overall,graduation_date
0,2,Tran Thi Lan,9.2,1,2026-04-02
1,7,Bui Tien Dung,9.2,1,2026-04-02
2,8,Ho Ngoc Mai,8.8,3,2026-04-04
3,5,Vu Quoc Anh,8.0,4,2026-04-05
4,3,Pham Van Nam,7.9,5,2026-04-06
5,4,Le Thanh Huyen,7.2,6,2026-04-07
6,9,Duong Huu Phuc,7.2,6,2026-04-07
7,10,Cao Thi Hanh,7.0,8,2026-04-09
8,1,Nguyen Minh Hoang,6.7,9,2026-04-10
9,6,Dang Thuy Linh,5.5,10,2026-04-11
